In [7]:
#複数の画像を1枚にまとめる
import cv2
import numpy as np
import os
from PIL import Image
import matplotlib.pyplot as plt
import random

In [ ]:
import cv2
from PIL import Image
import os
#画像回転
input_folder = "/home/data/0203_energee_after/makeimg/"

for i in os.listdir(input_folder):
    img_path = os.path.join(input_folder, i)
    
    # OpenCV で画像を読み込む（NumPy 配列）
    img = cv2.imread(img_path)

    # 画像サイズを確認
    if img.shape[0] == 1536 and img.shape[1] == 2048:
        continue  # すでに 2048x1536 ならスキップ

    print(f"Processing: {i}")

    # NumPy 配列を PIL 画像に変換
    img_pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    # 90度回転（時計回り）
    rotated_img = img_pil.rotate(-90, expand=True)

    # PIL 画像を保存
    rotated_img.save(img_path)

print("変換完了！")


In [1]:
import os
import random
from PIL import Image

def create_collage(input_folder, output_folder, num_outputs, cols=6, rows=4):
    """
    フォルダ内の画像をランダムに選び、リサイズせずに指定の列×行のグリッドで結合する。
    :param input_folder: 画像が保存されているフォルダ
    :param output_folder: 作成された画像を保存するフォルダ
    :param num_outputs: 作成するコラージュ画像の枚数
    :param cols: 横の画像数
    :param rows: 縦の画像数
    """
    os.makedirs(output_folder, exist_ok=True)
    
    image_files = [f for f in os.listdir(input_folder) if f.lower().endswith(('png', 'jpg', 'jpeg'))]

    if len(image_files) < cols * rows:
        raise ValueError("フォルダ内の画像数が足りません。")
    
    for i in range(num_outputs):
        selected_images = random.sample(image_files, cols * rows)
        images = [Image.open(os.path.join(input_folder, img)) for img in selected_images]

        # 画像のサイズを取得（最初の画像のサイズを基準にする）
        img_width, img_height = images[0].size

        # コラージュの全体サイズ
        collage_width = cols * img_width
        collage_height = rows * img_height

        # 空のキャンバスを作成
        collage = Image.new('RGB', (collage_width, collage_height))

        # 画像を貼り付ける
        for index, img in enumerate(images):
            x_offset = (index % cols) * img_width
            y_offset = (index // cols) * img_height
            collage.paste(img, (x_offset, y_offset))

        # 画像を保存
        output_path = os.path.join(output_folder, f'collage_{i+1}.jpg')
        collage.save(output_path)
        print(f'Saved: {output_path}')

# 使い方
input_folder = "/home/data/jikuari_maesyori/org/"  # 画像があるフォルダのパス
output_folder = '/home/data/jikuari_maesyori/hukusugazou'  # 結果を保存するフォルダのパス
num_outputs = 30  # 作成するコラージュ画像の数

create_collage(input_folder, output_folder, num_outputs)


Saved: /home/data/jikuari_maesyori/hukusugazou/collage_1.jpg
Saved: /home/data/jikuari_maesyori/hukusugazou/collage_2.jpg
Saved: /home/data/jikuari_maesyori/hukusugazou/collage_3.jpg
Saved: /home/data/jikuari_maesyori/hukusugazou/collage_4.jpg
Saved: /home/data/jikuari_maesyori/hukusugazou/collage_5.jpg
Saved: /home/data/jikuari_maesyori/hukusugazou/collage_6.jpg
Saved: /home/data/jikuari_maesyori/hukusugazou/collage_7.jpg
Saved: /home/data/jikuari_maesyori/hukusugazou/collage_8.jpg
Saved: /home/data/jikuari_maesyori/hukusugazou/collage_9.jpg
Saved: /home/data/jikuari_maesyori/hukusugazou/collage_10.jpg
Saved: /home/data/jikuari_maesyori/hukusugazou/collage_11.jpg
Saved: /home/data/jikuari_maesyori/hukusugazou/collage_12.jpg
Saved: /home/data/jikuari_maesyori/hukusugazou/collage_13.jpg
Saved: /home/data/jikuari_maesyori/hukusugazou/collage_14.jpg
Saved: /home/data/jikuari_maesyori/hukusugazou/collage_15.jpg
Saved: /home/data/jikuari_maesyori/hukusugazou/collage_16.jpg
Saved: /home/data

In [21]:
for i in os.listdir(input_folder):
    img_path = os.path.join(input_folder, i)
    if img.shape[0] == 1536 and img.shape[1] == 2048:
        continue
    print(f"違うサイズ: {i}")

In [ ]:
#6×4で最初の2列A等級，次の2列B等級，次の2列C等級（被りなし）

import os
import random
from PIL import Image

def create_collage_column_sorted(folder_dict, output_folder, cols_per_grade=2, rows=4):
    """
    コラージュを列単位で AABBCC の順で作る（重複なし）

    :param folder_dict: {'A': path, 'B': path, 'C': path}
    :param output_folder: 保存先
    :param cols_per_grade: 各等級の列数（例：2列ずつ）
    :param rows: 行数（例：4行）
    """
    os.makedirs(output_folder, exist_ok=True)
    grades = ['A', 'B', 'C']
    total_cols = cols_per_grade * len(grades)
    total_images_per_grade = cols_per_grade * rows

    # 各等級の画像を準備
    image_pool = {}
    for grade in grades:
        files = [f for f in os.listdir(folder_dict[grade]) if f.lower().endswith(('png', 'jpg', 'jpeg'))]
        random.shuffle(files)
        image_pool[grade] = files

    i = 1
    while all(len(image_pool[g]) >= total_images_per_grade for g in grades):
        collage_images = []

        # 各行に対してA→B→Cの順で2枚ずつ配置
        for row in range(rows):
            for grade in grades:
                selected = image_pool[grade][:cols_per_grade]
                image_pool[grade] = image_pool[grade][cols_per_grade:]
                collage_images.extend([Image.open(os.path.join(folder_dict[grade], f)) for f in selected])

        # サイズ取得とコラージュ作成
        img_width, img_height = collage_images[0].size
        collage_width = total_cols * img_width
        collage_height = rows * img_height
        collage = Image.new('RGB', (collage_width, collage_height))

        for index, img in enumerate(collage_images):
            row = index // total_cols
            col = index % total_cols
            x_offset = col * img_width
            y_offset = row * img_height
            collage.paste(img, (x_offset, y_offset))

        output_path = os.path.join(output_folder, f'collage_{i}.jpg')
        collage.save(output_path)
        print(f"Saved: {output_path}")
        i += 1

    print("画像が足りなくなったため終了。")

    
    
folder_dict = {
    'A': "/home/data/jikuari_maesyori_gakusyu/org/A",
    'B': "/home/data/jikuari_maesyori_gakusyu/org/B",
    'C': "/home/data/jikuari_maesyori_gakusyu/org/C"
}
output_folder = "/home/data/jikuari_maesyori_gakusyu/hukusugazou_aabbcc"

create_collage_column_sorted(folder_dict, output_folder)


Saved: /home/data/jikuari_maesyori_gakusyu/hukusugazou_aabbcc/collage_1.jpg
Saved: /home/data/jikuari_maesyori_gakusyu/hukusugazou_aabbcc/collage_2.jpg
Saved: /home/data/jikuari_maesyori_gakusyu/hukusugazou_aabbcc/collage_3.jpg
Saved: /home/data/jikuari_maesyori_gakusyu/hukusugazou_aabbcc/collage_4.jpg
Saved: /home/data/jikuari_maesyori_gakusyu/hukusugazou_aabbcc/collage_5.jpg
画像が足りなくなったため終了。


Processing: collage_1.jpg

image 1/1 /home/data/jikuari_maesyori_gakusyu/hukusugazou_aabbcc/collage_1.jpg: 320x640 25 shiitake_bboxs, 19.9ms
Speed: 1.5ms preprocess, 19.9ms inference, 0.3ms postprocess per image at shape (1, 3, 320, 640)

0: 640x512 1 shiitake, 25.1ms
Speed: 0.8ms preprocess, 25.1ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 512)

0: 448x640 1 shiitake, 26.7ms
Speed: 0.9ms preprocess, 26.7ms inference, 0.5ms postprocess per image at shape (1, 3, 448, 640)

0: 640x640 1 shiitake, 23.8ms
Speed: 1.8ms preprocess, 23.8ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x576 1 shiitake, 22.1ms
Speed: 1.1ms preprocess, 22.1ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 576)

0: 640x640 1 shiitake, 20.6ms
Speed: 1.1ms preprocess, 20.6ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 shiitake, 20.2ms
Speed: 1.1ms preprocess, 20.2ms inference, 0.6ms postprocess per image at shape (1, 3, 640,